In [146]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler , OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Data loading and cleaning

1. load the dataset

In [147]:
df = pd.read_csv("..\\data\\WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [148]:
df.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


2. Totalcharges column text to numeric and remove the rows with empty values

In [149]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)

3. change the target column to binary values

In [150]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [151]:
print("Dataset Shape:", df.shape)
df[['tenure', 'MonthlyCharges', 'TotalCharges','Contract', 'Churn']].head(5)

Dataset Shape: (7032, 21)


,tenure,MonthlyCharges,TotalCharges,Contract,Churn
0,1,29.85,29.85,Month-to-month,0
1,34,56.95,1889.50,One year,0
2,2,53.85,108.15,Month-to-month,1
3,45,42.30,1840.75,One year,0
4,2,70.70,151.65,Month-to-month,1


# Preprocessing pipeline and Model training

1. choose main 8 features for Streamlit app

In [152]:
num_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_features = ['Contract', 'InternetService', 'OnlineSecurity', 'TechSupport', 'PaymentMethod']

X = df[num_features + cat_features]
y = df['Churn']

2. use split before scaling to avoid data leakage

In [153]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

3. Preprocessor that scales the numeric features and encodes the categorical features

In [154]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ]
)

4. Full Pipeline (Preprocessor + Random Forest Model)

In [155]:
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=8,
        class_weight='balanced',
        random_state=42
    ))
])

5. Train the model 

In [156]:
model_pipeline.fit(X_train, y_train)
print("Model Training is completed.")

Model Training is completed.


# Evaluate the Model and Save it

1. Test results using test data

In [157]:
y_pred = model_pipeline.predict(X_test)
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

Classification Report:

              precision    recall  f1-score   support

           0       0.90      0.76      0.82      1033
           1       0.53      0.76      0.63       374

    accuracy                           0.76      1407
   macro avg       0.71      0.76      0.72      1407
weighted avg       0.80      0.76      0.77      1407



2. Save the trained pipeline to the models/ folder for later use in the Streamlit app

In [158]:
joblib.dump(model_pipeline, '../models/churn_pipeline.pkl')
print("Saved successfully: models/churn_pipeline.pkl")

Saved successfully: models/churn_pipeline.pkl
